# Classes, tree-sitter module: guava (java)
Module: `core-module/classes.py`. Each rule must show the rows it fixes; a rule that fixes nothing here is still listed with 0.

In [1]:
import pandas as pd, class_lab as L
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 250)
APP, LANG = 'guava', 'java'
REPO = L.ROOT / 'apps' / APP
DB = L.ROOT / 'results' / APP / 'runs/iter-003_2026-11-17_2247/crg/graph.db'
files = L.repo_files(REPO, LANG)
print(LANG, 'files:', len(files))

java files: 3267


## 1. Reference: javac (`java-ref/ListClasses.java`)
Named classes are expected. Anonymous classes have no name and are not expected.

In [2]:
ref = L.reference(REPO, LANG, files)
print('parse errors:', ref.error.notna().sum())
ref = ref[ref.error.isna()]
print(ref.kind.value_counts().to_string())
named = ref[ref.kind != 'anonymous'].reset_index(drop=True)
print('expected named classes:', len(named))

parse errors: 0
kind
class         6416
anonymous     4444
interface      452
enum           392
annotation     140
expected named classes: 7400


## 2. Module with no rules
The plain tree-sitter result, before any rule.

In [3]:
m0, m0_miss, m0_extra = L.score(named, L.module(REPO, LANG, files, rules=frozenset()))
pd.Series(m0).to_frame('module, no rules')

,"module, no rules"
expected,7400.0
found,7260.0
found %,98.1
same end line,7260.0
missed,140.0
extra,0.0


In [4]:
print('missed'); display(m0_miss.assign(source=[L.source_line(REPO, f, l) for f, l in zip(m0_miss.file, m0_miss.line)]))
print('extra'); display(m0_extra.assign(source=[L.source_line(REPO, f, l) for f, l in zip(m0_extra.file, m0_extra.line_start)]))

missed


,file,name,line,end,kind,source
37,android/guava-testlib/src/com/google/common/collect/testing/IgnoreJRERequirement.java,IgnoreJRERequirement,30,32,annotation,"@Target({METHOD, CONSTRUCTOR, TYPE, FIELD})"
119,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionFeature.java,Require,118,125,annotation,@Retention(RetentionPolicy.RUNTIME)
121,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionSize.java,Require,90,97,annotation,@Retention(RetentionPolicy.RUNTIME)
126,android/guava-testlib/src/com/google/common/collect/testing/features/ListFeature.java,Require,60,67,annotation,@Retention(RetentionPolicy.RUNTIME)
128,android/guava-testlib/src/com/google/common/collect/testing/features/MapFeature.java,Require,87,94,annotation,@Retention(RetentionPolicy.RUNTIME)
...,...,...,...,...,...,...
7143,guava/src/com/google/common/reflect/ParametricNullness.java,ParametricNullness,69,72,annotation,@GwtCompatible
7309,guava/src/com/google/common/util/concurrent/IgnoreJRERequirement.java,IgnoreJRERequirement,29,30,annotation,"@Target({METHOD, CONSTRUCTOR, TYPE, FIELD})"
7338,guava/src/com/google/common/util/concurrent/ParametricNullness.java,ParametricNullness,69,72,annotation,@GwtCompatible
7340,guava/src/com/google/common/util/concurrent/Partially.java,GwtIncompatible,41,46,annotation,@Retention(RetentionPolicy.CLASS)


extra


,file,name,line_start,line_end,source


## 3. Proof per rule
For each rule: the module is run with every rule **except** that one, then with all rules. The rows that change are the rows the rule fixes.

In [5]:
rules, fixed = L.rule_proof(REPO, LANG, files, named)
rules

,rule,what,found without rule,found with all rules,rows fixed (missed -> found),wrong rows removed
0,1,java: annotation types (@interface) are classes,7260,7400,140,0
1,2,js: class expression named by the variable/property it is assigned to,7400,7400,0,0
2,3,python: decorated class starts at its first decorator,7400,7400,0,0


In [6]:
fixed

,rule,fix,file,name,line,source
0,1,missed -> found,android/guava-testlib/src/com/google/common/collect/testing/IgnoreJRERequirement.java,IgnoreJRERequirement,30,"@Target({METHOD, CONSTRUCTOR, TYPE, FIELD})"
1,1,missed -> found,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionFeature.java,Require,118,@Retention(RetentionPolicy.RUNTIME)
2,1,missed -> found,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionSize.java,Require,90,@Retention(RetentionPolicy.RUNTIME)
3,1,missed -> found,android/guava-testlib/src/com/google/common/collect/testing/features/ListFeature.java,Require,60,@Retention(RetentionPolicy.RUNTIME)
4,1,missed -> found,android/guava-testlib/src/com/google/common/collect/testing/features/MapFeature.java,Require,87,@Retention(RetentionPolicy.RUNTIME)
...,...,...,...,...,...,...
135,1,missed -> found,guava/src/com/google/common/reflect/ParametricNullness.java,ParametricNullness,69,@GwtCompatible
136,1,missed -> found,guava/src/com/google/common/util/concurrent/IgnoreJRERequirement.java,IgnoreJRERequirement,29,"@Target({METHOD, CONSTRUCTOR, TYPE, FIELD})"
137,1,missed -> found,guava/src/com/google/common/util/concurrent/ParametricNullness.java,ParametricNullness,69,@GwtCompatible
138,1,missed -> found,guava/src/com/google/common/util/concurrent/Partially.java,GwtIncompatible,41,@Retention(RetentionPolicy.CLASS)


## 4. Module with all rules

In [7]:
m, m_miss, m_extra = L.score(named, L.module(REPO, LANG, files))
pd.DataFrame({'no rules': m0, 'all rules': m})

,no rules,all rules
expected,7400.0,7400.0
found,7260.0,7400.0
found %,98.1,100.0
same end line,7260.0,7400.0
missed,140.0,0.0
extra,0.0,0.0


In [8]:
print('still missed'); display(m_miss)
print('still extra'); display(m_extra)

still missed


,file,name,line,end,kind


still extra


,file,name,line_start,line_end


## 5. False positives
**False positive** = a module class whose name is not a class in that file in the reference (any line).

In [9]:
m_all = L.module(REPO, LANG, files)
m_fp = L.false_positives(named, m_all, REPO)
print('false positives:', len(m_fp))
m_fp

false positives: 0


,file,name,line_start,line_end,source
